# Explore a near-optimal set

This notebook explores the near-optimal designs of PyPSA's `model_energy` example: a
single-node system with wind, solar, battery and hydrogen storage. It starts from the
cheapest design, asks for more wind power, looks at the cost of getting there, and at what
changes in the system along the path.

It uses the model's **near-optimal set** published with the repository (4.3 MB), which it
downloads on first run. Run it from the `mga-compass` directory with the `notebook` extra,
for plots:

```bash
uv run --extra notebook --with jupyter jupyter lab
```

To explore a set of your own instead, set `SET` below to its directory, its zip archive or
its URL. The dataset script builds this one (about 10 minutes; see the *Dataset scripts*
reference):

```bash
uv run --extra scripts python scripts/generate_dataset.py --tol 5 --max-iter 120 --max-parallel 4
```

In [ ]:
import pandas as pd

import mga_compass as mc

pd.set_option("display.float_format", "{:,.1f}".format)

SET = "https://github.com/EINS-TUDa/MGA-Compass/releases/download/dataset-model-energy-2026-11-07/near-optimal-set.zip"
DIRECTORY = "data/model-energy"  # where the set is unpacked; its outputs are read from here

## Load the set

`mc.load` downloads the set, unpacks it into `DIRECTORY` and checks it. The result is a
`Space`: the set's designs in the space of its properties of interest (PoIs), here the
capacity of each technology, plus their cost. Values are in the set's display units
(capacities in GW and TWh, the cost in M€). By default, the space is the largest slack level of the set.

In [ ]:
space = mc.load(SET, DIRECTORY)
space

In [ ]:
space.points.head()

Each PoI's range over the designs, with the optimum marked:

In [ ]:
space.plot_ranges();

## Ask for more wind

Start from the cheapest design and ask for 20% more wind capacity, while the cost may rise by
at most 3%. The order of the requests is their priority: wind first, then cost. Solar,
battery and hydrogen are not listed, so they are free to change however is cheapest.

In [ ]:
result = space.navigate(space.optimum, {
    "wind": ("increase", "20%"),
    "TOTEX": ("keep", "3%"),
})
result

If a request cannot be met in full, it is met as far as possible, and the result says
how it was adjusted (`result.changes`). The PoIs at the start and the target, as bars and
as a radar chart, each PoI scaled to its range in the space:

In [ ]:
result.traverse.plot_pois();

In [ ]:
result.traverse.plot_radar();

The space projected onto two PoIs (or a PoI and the cost): the outline of the designs'
convex hull in that plane, the designs, and the path from start to target. Designs inside the
outline lie deeper along the other PoIs.

In [ ]:
space.plot_projection("wind", "solar", result);

## The path

`result.traverse` is the path from the start to the target: the PoIs move in a straight line,
and the cost along the path is the cheapest mix of the set's designs. The designs' duals give
a lower bound: the true cost of the designs on the path lies between the two.

In [ ]:
result.traverse

The plot also shows the optimal cost and the budget of the space, the optimal cost plus
its slack; `beta` marks a design on the path.

In [ ]:
result.traverse.plot(beta=0.5);

The PoIs of that design, with lines at the start's (green) and the target's (red)
values; the bars move between them as `beta` goes from 0 to 1. `plot_radar` takes a `beta`
too.

In [ ]:
result.traverse.plot_pois(beta=0.5);

That design is a mix of the set's designs. `weights_at` lists them, heaviest first, with
the slack level and the kind of search each was found by, and its `key`: the hash in the name
of its result file, e.g. `slack_0.02/mga_furthest_result_<key>.nc` for a box design, to go
back to the full network.

In [ ]:
with pd.option_context("display.float_format", "{:.3g}".format):  # weights and slacks are small
    display(result.traverse.weights_at(0.5))

## What changes in the system

Every design here is a mix of the set's designs, and its detailed outputs are the same mix of
theirs. `space.outputs()` lists the outputs; `space.output` gives one for a design: a
navigation result (its target), the weights of any point along the traverse, or a design of
the set by its id.

In [ ]:
space.outputs()

In [ ]:
pd.DataFrame({
    "start": space.output("capacity", result.traverse.alpha_at(0)),
    "halfway": space.output("capacity", result.traverse.alpha_at(0.5)),
    "target": space.output("capacity", result),
})

The set also declares plots of its outputs, as the web platform shows them, with its
labels and colours. `space.plots()` lists them; `space.plot` draws one for a design:

In [ ]:
space.plots()

In [ ]:
space.plot("generation", result);

In [ ]:
space.plot("capacity", result);

## Carry on

Navigation can start from any result. From the design with more wind, ask for at least 50%
more hydrogen storage, keeping wind where it is and the cost within 2% of the start's.

After meeting every request, navigation improves them one by one, in their order: an
increase goes as far as the requests before it allow, a keep stays as close as they allow.
With hydrogen before the cost, hydrogen grows until the cost reaches its 2% limit; with the
cost first, the cost would stay as close to the start's as possible, and hydrogen would grow
by just the 50%.

In [ ]:
space.navigate(result, {
    "wind": "keep",
    "hydrogen": ("increase", "50%"),
    "TOTEX": ("keep", "2%"),
})

## A tighter budget

`slack` picks a smaller slack level of the set: only the designs found within that much of
the optimal cost. The set is unpacked already, so it loads from its directory.

In [ ]:
tight = mc.load(DIRECTORY, slack=0.02)
tight.navigate(tight.optimum, {"wind": ("increase", "20%")})